# Create user density map (Figure 3)

This notebook creates a world map showing density of users by country.

Requirements:

- MaxMind GeoLite2-City file, downloaded from: https://www.maxmind.com/en/accounts/1185920/geoip/downloads
- List of registration IP addresses from PhysioNet (download the users spreadsheet from the console).

# Setup

In [ ]:
pip install pandas geoip2 geopandas iso3166 matplotlib geodatasets


In [ ]:
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from iso3166 import countries
from matplotlib.colors import LogNorm
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

In [ ]:
GEOIP_DB_PATH = "../data/physionet/GeoLite2-City.mmdb"
INPUT_CSV = "../data/physionet/users.csv"

COUNTRY_COUNTS_CSV = "../data/physionet/ip_country_counts.csv"

OUTPUT_FIG_PNG = "../figures/figure_3_world_map.png"
OUTPUT_FIG_SVG = "../figures/figure_3_world_map.svg"

In [ ]:
# -----------------------------
# Load country counts
# -----------------------------
counts = pd.read_csv(COUNTRY_COUNTS_CSV)

# ISO2 -> ISO3 (match Natural Earth)
def iso2_to_iso3(code):
    if pd.isna(code):
        return None
    try:
        return countries.get(code).alpha3
    except Exception:
        return None

counts["iso_a3"] = counts["country_iso2"].apply(iso2_to_iso3)

# -----------------------------
# Load world polygons (Natural Earth)
# -----------------------------
ne_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(ne_url)

# Normalize columns + create iso_a3 column consistent with counts
# Natural Earth uses ADM0_A3 as ISO3 for most countries
world = world.rename(columns={"ADM0_A3": "iso_a3", "ADMIN": "name"})

# Drop Antarctica
world = world[world["name"] != "Antarctica"]

# -----------------------------
# Merge counts onto world
# -----------------------------
world_counts = world.merge(counts[["iso_a3", "count"]], on="iso_a3", how="left")

# Ensure GeoDataFrame (critical: guarantees .boundary exists)
world_counts = gpd.GeoDataFrame(world_counts, geometry="geometry", crs=world.crs)

# Prepare plotting column
world_counts["count"] = world_counts["count"].fillna(0)
world_counts["count_plot"] = world_counts["count"].replace(0, np.nan)

has_users = world_counts.dropna(subset=["count_plot"])
max_count = float(has_users["count_plot"].max())

# Extend colour scale to next power of 10 above max_count
vmax = 10 ** math.ceil(math.log10(max_count))

# Generate world map

In [ ]:
# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(11, 5.5), constrained_layout=True)

# Base fill (no edges here)
world_counts.plot(ax=ax, color="#f2f2f2", linewidth=0)

# Borders (draw separately using line-friendly args)
world_counts.boundary.plot(ax=ax, color="white", linewidth=0.3)

# User countries fill
norm = LogNorm(vmin=1, vmax=vmax)
cmap = plt.cm.viridis

has_users.plot(column="count_plot", ax=ax, cmap=cmap, norm=norm, linewidth=0)

# User borders
has_users.boundary.plot(ax=ax, color="#444444", linewidth=0.4)

ax.set_axis_off()
ax.set_title("Global Distribution of Registered Users", fontsize=16, pad=12)

# -----------------------------
# Compact colorbar (bottom-right)
# -----------------------------
cax = inset_axes(
    ax,
    width="20%",
    height="2%",
    loc="lower right",
    borderpad=2,
)

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")

max_decade = int(math.log10(vmax))
ticks = [10 ** k for k in range(0, max_decade + 1)]
cbar.set_ticks(ticks)
cbar.set_ticklabels([f"{int(t)}" for t in ticks])
cbar.ax.tick_params(labelsize=8)
cbar.set_label("Unique IP addresses (log scale)", fontsize=9, labelpad=4)

plt.savefig(OUTPUT_FIG_PNG, dpi=300)
plt.savefig(OUTPUT_FIG_SVG)
plt.close(fig)